# Evaluate AutoDDG on the PANGAEA sample

This notebook evaluates AutoDDG descriptions with the same grounding and reference-similarity metrics used in `03-TvdP-validate-metadata.ipynb`. It reads AutoDDG outputs directly, excludes timeout records, preserves the existing model scores, and writes separate checkpoint and final files.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from tqdm.auto import tqdm

from langchain_openai import ChatOpenAI
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import FactualCorrectness, RubricsScore

ENV_PATH = Path('/home/com3dian/Github/metadata_agent/.env')
load_dotenv(ENV_PATH)

SURF_API_KEY = os.getenv('SURF_API_KEY')
SURF_API_BASE = os.getenv('SURF_API_BASE')
if not SURF_API_KEY or not SURF_API_BASE:
    raise RuntimeError(
        f'SURF_API_KEY or SURF_API_BASE is missing from {ENV_PATH}'
    )

REPO_ROOT = Path('/home/com3dian/Github/process_data_metadata_agent')
AUTODDG_OUTPUT_ROOT = Path('/home/com3dian/Github/metadata_agent/outputs')
DATASET_ROOT = REPO_ROOT / 'outputs' / 'pangaea' / 'pangaea_datasets'
ANNOTATION_PATH = REPO_ROOT / 'outputs' / 'pangaea' / 'sample_20260629_163144_filtered_top5_per_topic.csv'
BASE_SCORES_PATH = REPO_ROOT / 'outputs' / 'pangaea' / 'df_scores_pangaea_descriptive.csv'
CHECKPOINT_PATH = REPO_ROOT / 'outputs' / 'pangaea' / 'df_scores_autoddg_checkpoint.csv'
FINAL_SCORES_PATH = REPO_ROOT / 'outputs' / 'pangaea' / 'df_scores_pangaea_descriptive_with_autoddg.csv'


In [ ]:
# AutoDDG generates a description, but not the paper evaluator's name and keyword fields.
# Therefore, this notebook evaluates description only.
AUTODDG_RUNS = [
    'autoddg_pangaea_default-text-large',
    'autoddg_pangaea_Qwen_Qwen3.6-35B-A3B-FP8',
    'autoddg_pangaea_Sehyo_Qwen3.5-122B-A10B-NVFP4',
    'autoddg_pangaea_mistralai_Devstral-Small-2-24B-Instruct-2512',
    'autoddg_pangaea_mistralai_Mistral-Small-3.2-24B-Instruct-2506',
]

EXPECTED_DATASETS = 75
# False evaluates every available description and reports missing coverage.
# Timeout records are not converted into predictions or scores.
REQUIRE_COMPLETE_RUNS = False
JUDGE_MODEL = 'mistralai/Devstral-Small-2-24B-Instruct-2512'


In [ ]:
llm = LangchainLLMWrapper(
    ChatOpenAI(
        model=JUDGE_MODEL,
        api_key=SURF_API_KEY,
        base_url=SURF_API_BASE,
        temperature=0,
        max_tokens=2048,
    )
)


In [ ]:
def load_autoddg_predictions(folder_path: Path, model_name: str) -> pd.DataFrame:
    rows = []

    # This pattern excludes *_autoddg.skipped.json and run_log.jsonl.
    for path in sorted(folder_path.glob('*_autoddg.json')):
        with path.open('r') as f:
            result = json.load(f)

        description = result.get('description')
        if not isinstance(description, str) or not description.strip():
            raise ValueError(f'Missing description in {path}')

        rows.append({
            'id': int(result.get('dataset_id', path.name.split('_')[0])),
            'description': description.strip(),
            'model': model_name,
        })

    predictions = pd.DataFrame(rows)
    if predictions.empty:
        raise ValueError(f'No AutoDDG predictions found in {folder_path}')
    if predictions['id'].duplicated().any():
        duplicates = predictions.loc[predictions['id'].duplicated(), 'id'].tolist()
        raise ValueError(f'Duplicate dataset IDs in {folder_path}: {duplicates}')

    return predictions


def merge_predictions(predictions: pd.DataFrame, annotations: pd.DataFrame) -> pd.DataFrame:
    unknown_ids = set(predictions['id']) - set(annotations['id'])
    if unknown_ids:
        raise ValueError(f'Prediction IDs absent from annotations: {sorted(unknown_ids)}')

    selected_annotations = annotations[annotations['id'].isin(predictions['id'])]
    merged = predictions.merge(
        selected_annotations[['id', 'description']],
        on='id',
        how='inner',
        suffixes=('_pred', '_annot'),
        validate='one_to_one',
    )
    if len(merged) != len(predictions):
        raise ValueError('Predictions and annotations did not merge one-to-one')
    return merged


In [ ]:
def serialize_dataset(df: pd.DataFrame) -> str:
    schema = '\n'.join(f'- {column} ({dtype})' for column, dtype in df.dtypes.items())
    sample = df.head(5).to_markdown(index=False)
    return f'Columns:\n{schema}\n\nSample rows:\n{sample}\n\nRow count: {len(df)}'


async def score_description_grounding(scorer, row, verbose=False):
    dataset_path = DATASET_ROOT / f"{int(row['id'])}.csv"
    dataset_context = serialize_dataset(pd.read_csv(dataset_path))
    predicted_description = row['description_pred']

    if verbose:
        print(dataset_context)
        print(predicted_description)

    sample = SingleTurnSample(
        user_input=(
            'A model generated metadata for a dataset from the data alone.\n\n'
            f'DATASET:\n{dataset_context}\n\n'
            'Judge only whether the DESCRIPTION below is grounded in this data—'
            'whether its claims could be inferred from the data. Do not reward or '
            'penalize style, completeness, or omissions.'
        ),
        response=f"description: {predicted_description}",
    )
    score = await scorer.single_turn_ascore(sample)
    return (score - 1) / 4


async def score_description_similarity(scorer, row):
    sample = SingleTurnSample(
        response=row['description_pred'],
        reference=row['description_annot'],
    )
    return await scorer.single_turn_ascore(sample)


async def score_all_descriptions(llm, merged: pd.DataFrame, verbose=False):
    grounding_rubrics = {
        'score1_description': 'Makes claims that contradict the data or reference fields/values that do not exist in it.',
        'score2_description': 'Several claims are unsupported by the data or appear invented.',
        'score3_description': 'Mostly supportable, but contains at least one unsupported or speculative claim.',
        'score4_description': 'Every claim is supportable from the data; some claims are vague or generic.',
        'score5_description': 'Every claim is directly supported by the data and is specific to it.',
    }
    grounding_scorer = RubricsScore(rubrics=grounding_rubrics, llm=llm)
    similarity_scorer = FactualCorrectness(llm=llm, mode='precision', atomicity='low')

    grounding_scores = {}
    similarity_scores = {}
    for _, row in tqdm(merged.iterrows(), total=len(merged)):
        dataset_id = int(row['id'])
        try:
            grounding_scores[dataset_id] = await score_description_grounding(
                grounding_scorer, row, verbose=verbose
            )
            similarity_scores[dataset_id] = await score_description_similarity(
                similarity_scorer, row
            )
        except Exception as error:
            grounding_scores[dataset_id] = None
            similarity_scores[dataset_id] = None
            print(f'Row {dataset_id} failed: {error}')

    return grounding_scores, similarity_scores


In [ ]:
annotations = pd.read_csv(ANNOTATION_PATH)
if annotations['id'].duplicated().any():
    raise ValueError('Annotation IDs must be unique')

scores_input_path = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else BASE_SCORES_PATH
df_scores = pd.read_csv(scores_input_path)

coverage_rows = []
predictions_by_run = {}
for run_folder in AUTODDG_RUNS:
    model_slug = run_folder.removeprefix('autoddg_pangaea_')
    model_name = f'AutoDDG_{model_slug}'
    predictions = load_autoddg_predictions(AUTODDG_OUTPUT_ROOT / run_folder, model_name)
    predictions_by_run[run_folder] = predictions
    coverage_rows.append({
        'run': run_folder,
        'model': model_name,
        'predictions': len(predictions),
        'expected': EXPECTED_DATASETS,
        'coverage_percent': 100 * len(predictions) / EXPECTED_DATASETS,
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)

incomplete = coverage.loc[
    coverage['predictions'] != EXPECTED_DATASETS, ['run', 'predictions']
]
if not incomplete.empty:
    print('Incomplete runs (only available descriptions will be evaluated):')
    display(incomplete)
    if REQUIRE_COMPLETE_RUNS:
        raise ValueError(f'Incomplete AutoDDG runs:\n{incomplete.to_string(index=False)}')


## Run the evaluation

The preceding cell performs a no-cost coverage check. With `REQUIRE_COMPLETE_RUNS = False`, the following cell evaluates every available description and leaves timed-out datasets unscored.

In [ ]:
for run_folder in AUTODDG_RUNS:
    predictions = predictions_by_run[run_folder]
    model_name = predictions['model'].iloc[0]
    faith_column = f'faith_{model_name}'
    similarity_column = f'sim_{model_name}'

    prediction_ids = set(predictions['id'])
    model_rows = df_scores['id'].isin(prediction_ids)
    scores_complete = (
        faith_column in df_scores.columns
        and similarity_column in df_scores.columns
        and int(model_rows.sum()) == len(prediction_ids)
        and df_scores.loc[model_rows, [faith_column, similarity_column]].notna().all().all()
    )

    if scores_complete:
        print(f'Skipping {model_name}: all available predictions are already scored')
        continue

    print(f'Evaluating {model_name}: existing columns are incomplete or absent')
    merged = merge_predictions(predictions, annotations)
    grounding_scores, similarity_scores = await score_all_descriptions(llm, merged)

    n_grounding = sum(pd.notna(value) for value in grounding_scores.values())
    n_similarity = sum(pd.notna(value) for value in similarity_scores.values())
    if n_grounding == 0 or n_similarity == 0:
        raise RuntimeError(
            f'{model_name} produced no valid scores; checkpoint was not updated. '
            'Review the row failure messages above.'
        )

    df_scores[faith_column] = df_scores['id'].map(grounding_scores)
    df_scores[similarity_column] = df_scores['id'].map(similarity_scores)
    df_scores.to_csv(CHECKPOINT_PATH, index=False)
    print(f'Saved checkpoint: {CHECKPOINT_PATH}')


In [ ]:
# The last cell reloads the checkpoint and generates the final scores.
# This cell is intentionally kept as a no-op so only the last cell must be rerun.


In [ ]:
# Self-contained finalization cell: this can be rerun by itself.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path('/home/com3dian/Github/process_data_metadata_agent')
checkpoint_path = repo_root / 'outputs' / 'pangaea' / 'df_scores_autoddg_checkpoint.csv'
final_scores_path = repo_root / 'outputs' / 'pangaea' / 'df_scores_pangaea_descriptive_with_autoddg.csv'
expected_datasets = 75

if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

# Keep the checkpoint raw. It preserves which outputs were actually missing.
df_scores_raw = pd.read_csv(checkpoint_path)
if len(df_scores_raw) != expected_datasets:
    raise ValueError(
        f'Expected {expected_datasets} score rows, found {len(df_scores_raw)}'
    )

faith_columns = sorted(
    column for column in df_scores_raw.columns
    if column.startswith('faith_AutoDDG_')
)
if not faith_columns:
    raise ValueError('No AutoDDG scores were found in the checkpoint')

model_names = [column.removeprefix('faith_') for column in faith_columns]
autoddg_score_columns = []
for model_name in model_names:
    faith_column = f'faith_{model_name}'
    similarity_column = f'sim_{model_name}'
    if similarity_column not in df_scores_raw.columns:
        raise ValueError(f'Missing checkpoint column: {similarity_column}')
    autoddg_score_columns.extend([faith_column, similarity_column])

# End-to-end scoring policy: every missing AutoDDG output or score counts as zero.
df_scores_final = df_scores_raw.copy()
df_scores_final[autoddg_score_columns] = (
    df_scores_final[autoddg_score_columns].fillna(0.0)
)
df_scores_final.to_csv(final_scores_path, index=False)
print(f'Saved final scores: {final_scores_path}')

summary_rows = []
for model_name in model_names:
    faith_column = f'faith_{model_name}'
    similarity_column = f'sim_{model_name}'
    n_available = int(
        df_scores_raw[[faith_column, similarity_column]].notna().all(axis=1).sum()
    )
    summary_rows.append({
        'model': model_name,
        'available_outputs': n_available,
        'missing_as_zero': expected_datasets - n_available,
        'denominator': expected_datasets,
        'faith_mean': df_scores_final[faith_column].mean(),
        'faith_sem': df_scores_final[faith_column].sem(),
        'similarity_mean': df_scores_final[similarity_column].mean(),
        'similarity_sem': df_scores_final[similarity_column].sem(),
    })

df_summary = pd.DataFrame(summary_rows).sort_values('model').reset_index(drop=True)
display(df_summary)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
x = np.arange(len(df_summary))
labels = df_summary['model'].str.removeprefix('AutoDDG_').str.replace('_', ' ', regex=False)

for axis, mean_column, sem_column, title in [
    (axes[0], 'faith_mean', 'faith_sem', 'Mean faithfulness (missing = 0)'),
    (axes[1], 'similarity_mean', 'similarity_sem', 'Mean similarity (missing = 0)'),
]:
    axis.bar(x, df_summary[mean_column], yerr=df_summary[sem_column], capsize=3)
    axis.set_title(title)
    axis.set_ylim(0, 1)
    axis.set_ylabel('Score (0–1), denominator = 75')
    axis.set_xticks(x, labels, rotation=45, ha='right')

fig.tight_layout()
